In [ ]:
# RQ3: Effect of Preprocessing Strategies
# How do different data preprocessing strategies affect model performance?

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/marketing-and-product-performance-dataset/marketing_and_product_performance.csv')
for col in ['Subscription_Tier', 'Common_Keywords']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df = df.drop(columns=['Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID'])
X = df.drop(columns=['Units_Sold'])
y = df['Units_Sold']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
strategies = {
    'No Scaling': (X_train.values, X_test.values),
    'StandardScaler': (StandardScaler().fit_transform(X_train), StandardScaler().fit(X_train).transform(X_test)),
    'MinMaxScaler': (MinMaxScaler().fit_transform(X_train), MinMaxScaler().fit(X_train).transform(X_test)),
}

results = []
for name, (Xtr, Xte) in strategies.items():
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    results.append({
        'Preprocessing': name,
        'MAE': round(mean_absolute_error(y_test, preds),4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)),4),
        'R2': round(r2_score(y_test, preds),4)
    })

res_df = pd.DataFrame(results)
print(res_df)
res_df.to_csv('RQ3_preprocessing_effect.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(res_df))
w = 0.25
ax.bar(x-w, res_df['MAE'], w, label='MAE', color='#2196F3')
ax.bar(x, res_df['RMSE'], w, label='RMSE', color='#FF9800')
ax.bar(x+w, res_df['R2']*100, w, label='R²(×100)', color='#4CAF50')
ax.set_xticks(x)
ax.set_xticklabels(res_df['Preprocessing'])
ax.set_ylabel('Score')
ax.set_title('RQ3: Effect of Preprocessing Strategies on Random Forest', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('RQ3_preprocessing_effect.pdf', dpi=150, bbox_inches='tight')
plt.show()